# Ingestión del archivo `movie_genre.json`

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

## 1. Leer el archivo JSON usando `DataFrameReader` de Spark

In [0]:
country_schema = "movieId INT, genreId INT"

movie_genres_df = (spark.read 
    .schema(country_schema)
    .json(f"{bronze_folder_path}/{v_file_date}/movie_genre.json")
)
display(movie_genres_df)

movieId,genreId
5,35
5,80
11,12
11,28
11,878
12,16
12,10751
13,18
13,35
13,10749


## 2. Cambiar el nombre de las columnas según lo requerido

In [0]:
movie_genres_renamed_df = (movie_genres_df
    .withColumnRenamed("movieId", "movie_id")
    .withColumnRenamed("genreId", "genre_id")
)

## 4. Agregar las columnas `ingestion_date` y `environmate` al DateFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit

movie_genres_final_df = add_ingestion_date(movie_genres_renamed_df).withColumn("enviroment", lit(v_environment)).withColumn("file_date", lit(v_file_date))

## 5. Escribir datos en el datalake en formato `Parquet`

In [0]:
merge_delta_lake( movie_genres_final_df, "movie_silver", "movies_genres", "tgt.movie_id = src.movie_id AND tgt.file_date = src.file_date AND tgt.genre_id = src.genre_id", "file_date" )

In [0]:
%sql
SELECT * FROM movie_silver.movies_genres

path,name,size,modificationTime
abfss://silver@moviehistory4.dfs.core.windows.net/movie_genres/_SUCCESS,_SUCCESS,0,1789060435000
abfss://silver@moviehistory4.dfs.core.windows.net/movie_genres/_committed_677141032324381480,_committed_677141032324381480,232,1789060434000
abfss://silver@moviehistory4.dfs.core.windows.net/movie_genres/_committed_8032520097671505623,_committed_8032520097671505623,124,1789059449000
abfss://silver@moviehistory4.dfs.core.windows.net/movie_genres/_started_677141032324381480,_started_677141032324381480,0,1789060432000
abfss://silver@moviehistory4.dfs.core.windows.net/movie_genres/_started_8032520097671505623,_started_8032520097671505623,0,1789059448000
abfss://silver@moviehistory4.dfs.core.windows.net/movie_genres/movie_id=10008/,movie_id=10008/,0,1789060097000
abfss://silver@moviehistory4.dfs.core.windows.net/movie_genres/movie_id=10025/,movie_id=10025/,0,1789060097000
abfss://silver@moviehistory4.dfs.core.windows.net/movie_genres/movie_id=100275/,movie_id=100275/,0,1789060315000
abfss://silver@moviehistory4.dfs.core.windows.net/movie_genres/movie_id=10028/,movie_id=10028/,0,1789059654000
abfss://silver@moviehistory4.dfs.core.windows.net/movie_genres/movie_id=10029/,movie_id=10029/,0,1789060098000
